In [0]:
print("Cloud Sales Analytics")
print("Meu primeiro projeto com Databricks!")
print("Ambiente Python funcionando.")

Cloud Sales Analytics
Meu primeiro projeto com Databricks!
Ambiente Python funcionando.


In [0]:
import pandas as pd

dados_vendas = [
    (1, "2026-01-05", "Notebook", "Tecnologia", "Sudeste", 2, 3500.00),
    (2, "2026-01-06", "Mouse", "Acessorios", "Sul", 5, 120.00),
    (3, "2026-01-07", "Teclado", "Acessorios", "Sudeste", 3, 250.00),
    (4, "2026-01-08", "Monitor", "Tecnologia", "Nordeste", 2, 1100.00),
    (5, "2026-01-10", "Notebook", "Tecnologia", "Sul", 1, 3500.00),
    (6, "2026-01-12", "Mouse", "Acessorios", "Sudeste", 10, 120.00),
    (7, "2026-01-15", "Cadeira", "Moveis", "Centro-Oeste", 2, 800.00),
    (8, "2026-01-18", "Monitor", "Tecnologia", "Sul", 3, 1100.00),
    (9, "2026-01-20", "Teclado", "Acessorios", "Nordeste", 4, 250.00),
    (10, "2026-01-22", "Cadeira", "Moveis", "Sudeste", 1, 800.00)
]

colunas = [
    "id_venda",
    "data_venda",
    "produto",
    "categoria",
    "regiao",
    "quantidade",
    "preco_unitario"
]

df_vendas = pd.DataFrame(dados_vendas, columns=colunas)

display(df_vendas)

id_venda,data_venda,produto,categoria,regiao,quantidade,preco_unitario
1,2026-01-05,Notebook,Tecnologia,Sudeste,2,3500.0
2,2026-01-06,Mouse,Acessorios,Sul,5,120.0
3,2026-01-07,Teclado,Acessorios,Sudeste,3,250.0
4,2026-01-08,Monitor,Tecnologia,Nordeste,2,1100.0
5,2026-01-10,Notebook,Tecnologia,Sul,1,3500.0
6,2026-01-12,Mouse,Acessorios,Sudeste,10,120.0
7,2026-01-15,Cadeira,Moveis,Centro-Oeste,2,800.0
8,2026-01-18,Monitor,Tecnologia,Sul,3,1100.0
9,2026-01-20,Teclado,Acessorios,Nordeste,4,250.0
10,2026-01-22,Cadeira,Moveis,Sudeste,1,800.0


In [0]:
df_vendas_spark = spark.createDataFrame(df_vendas)

display(df_vendas_spark)

id_venda,data_venda,produto,categoria,regiao,quantidade,preco_unitario
1,2026-01-05,Notebook,Tecnologia,Sudeste,2,3500.0
2,2026-01-06,Mouse,Acessorios,Sul,5,120.0
3,2026-01-07,Teclado,Acessorios,Sudeste,3,250.0
4,2026-01-08,Monitor,Tecnologia,Nordeste,2,1100.0
5,2026-01-10,Notebook,Tecnologia,Sul,1,3500.0
6,2026-01-12,Mouse,Acessorios,Sudeste,10,120.0
7,2026-01-15,Cadeira,Moveis,Centro-Oeste,2,800.0
8,2026-01-18,Monitor,Tecnologia,Sul,3,1100.0
9,2026-01-20,Teclado,Acessorios,Nordeste,4,250.0
10,2026-01-22,Cadeira,Moveis,Sudeste,1,800.0


In [0]:
df_vendas_spark.printSchema()

root
 |-- id_venda: long (nullable = true)
 |-- data_venda: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- quantidade: long (nullable = true)
 |-- preco_unitario: double (nullable = true)



In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+



In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+



In [0]:
df_vendas_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_vendas")

In [0]:
df_bronze = spark.table("workspace.default.bronze_vendas")

display(df_bronze)

id_venda,data_venda,produto,categoria,regiao,quantidade,preco_unitario
1,2026-01-05,Notebook,Tecnologia,Sudeste,2,3500.0
2,2026-01-06,Mouse,Acessorios,Sul,5,120.0
3,2026-01-07,Teclado,Acessorios,Sudeste,3,250.0
4,2026-01-08,Monitor,Tecnologia,Nordeste,2,1100.0
5,2026-01-10,Notebook,Tecnologia,Sul,1,3500.0
6,2026-01-12,Mouse,Acessorios,Sudeste,10,120.0
7,2026-01-15,Cadeira,Moveis,Centro-Oeste,2,800.0
8,2026-01-18,Monitor,Tecnologia,Sul,3,1100.0
9,2026-01-20,Teclado,Acessorios,Nordeste,4,250.0
10,2026-01-22,Cadeira,Moveis,Sudeste,1,800.0


In [0]:
from pyspark.sql.functions import col, to_date

df_silver = (
    df_bronze
    .withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))
    .withColumn("valor_total", col("quantidade") * col("preco_unitario"))
)

display(df_silver)

id_venda,data_venda,produto,categoria,regiao,quantidade,preco_unitario,valor_total
1,2026-01-05,Notebook,Tecnologia,Sudeste,2,3500.0,7000.0
2,2026-01-06,Mouse,Acessorios,Sul,5,120.0,600.0
3,2026-01-07,Teclado,Acessorios,Sudeste,3,250.0,750.0
4,2026-01-08,Monitor,Tecnologia,Nordeste,2,1100.0,2200.0
5,2026-01-10,Notebook,Tecnologia,Sul,1,3500.0,3500.0
6,2026-01-12,Mouse,Acessorios,Sudeste,10,120.0,1200.0
7,2026-01-15,Cadeira,Moveis,Centro-Oeste,2,800.0,1600.0
8,2026-01-18,Monitor,Tecnologia,Sul,3,1100.0,3300.0
9,2026-01-20,Teclado,Acessorios,Nordeste,4,250.0,1000.0
10,2026-01-22,Cadeira,Moveis,Sudeste,1,800.0,800.0


In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_vendas")

In [0]:
df_silver_salvo = spark.table("workspace.default.silver_vendas")

display(df_silver_salvo)

id_venda,data_venda,produto,categoria,regiao,quantidade,preco_unitario,valor_total
1,2026-01-05,Notebook,Tecnologia,Sudeste,2,3500.0,7000.0
2,2026-01-06,Mouse,Acessorios,Sul,5,120.0,600.0
3,2026-01-07,Teclado,Acessorios,Sudeste,3,250.0,750.0
4,2026-01-08,Monitor,Tecnologia,Nordeste,2,1100.0,2200.0
5,2026-01-10,Notebook,Tecnologia,Sul,1,3500.0,3500.0
6,2026-01-12,Mouse,Acessorios,Sudeste,10,120.0,1200.0
7,2026-01-15,Cadeira,Moveis,Centro-Oeste,2,800.0,1600.0
8,2026-01-18,Monitor,Tecnologia,Sul,3,1100.0,3300.0
9,2026-01-20,Teclado,Acessorios,Nordeste,4,250.0,1000.0
10,2026-01-22,Cadeira,Moveis,Sudeste,1,800.0,800.0


In [0]:
from pyspark.sql.functions import sum, round

df_gold = (
    df_silver_salvo
    .groupBy("categoria")
    .agg(
        round(sum("valor_total"), 2).alias("faturamento_total")
    )
    .orderBy("categoria")
)

display(df_gold)

categoria,faturamento_total
Acessorios,3550.0
Moveis,2400.0
Tecnologia,16000.0


In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_faturamento_categoria")

In [0]:
df_gold_salvo = spark.table("workspace.default.gold_faturamento_categoria")

display(df_gold_salvo)

categoria,faturamento_total
Acessorios,3550.0
Moveis,2400.0
Tecnologia,16000.0


In [0]:
%sql
SELECT
    categoria,
    SUM(valor_total) AS faturamento_total
FROM workspace.default.silver_vendas
GROUP BY categoria
ORDER BY faturamento_total DESC;

categoria,faturamento_total
Tecnologia,16000.0
Acessorios,3550.0
Moveis,2400.0


In [0]:
%sql
SELECT
    ROUND(SUM(valor_total), 2) AS faturamento_total_geral,
    COUNT(*) AS quantidade_vendas,
    ROUND(AVG(valor_total), 2) AS ticket_medio
FROM workspace.default.silver_vendas;

faturamento_total_geral,quantidade_vendas,ticket_medio
21950.0,10,2195.0


In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_vendas,false
default,gold_faturamento_categoria,false
default,silver_vendas,false
